# EEG Denoising

1. Notch filter with (50Hz)60Hz and bandpass filter with 0.5(-1)-30Hz(-50Hz) → this removes powerline noise and generally removes all frequencies that are unwanted in our data (This is done before the segmentation because of the 5s break in between)
2. Remove Artificats with ICA
3. Remove extremely high amplitudes with Amplitude based epoch rejection
4. Normalize data by applying a z-transform
5. Visual inspection


## 0. Load the data

In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
PATH_TO_DATA = "../../data/eeg_data_v3_denoised"
PATH_TO_DATA_CLEANED = "../../data/eeg_data_v5_denoised"
PATH_TO_DATA_CLEANED_2 = "../../data/eeg_data_normalized_features_noCh3_lessStrict_reduced"

In [3]:
df = pd.read_pickle(f"{PATH_TO_DATA}/pickle/001.pkl")
df_cleaned = pd.read_pickle(f"{PATH_TO_DATA_CLEANED}/pickle/001.pkl")
df_cleaned_2 = pd.read_pickle(f"{PATH_TO_DATA_CLEANED_2}/pickle/001.pkl")


In [4]:
df_cleaned_2

,genre,EEG_Ch1,EEG_Ch2,EEG_Ch4,EEG_Ch5,EEG_Ch6,EEG_Ch7,EEG_Ch8
Song0,disco,"[0.46996475649271563, 0.46644168472467457, 0.3...","[1.7448091337906189, 1.7735713409220055, 1.696...","[-0.4281515374564015, -0.2093011672581785, -0....","[0.7193737470282048, 0.7353037264244446, 0.546...","[-0.3597482866641765, -0.2474780794589911, -0....","[1.0683068651379781, 1.1598938679590518, 1.017...","[0.8656230170004618, 0.9766435595630324, 0.856..."
Song1,rock,"[-1.7511517172750644, -1.7034418493755337, -1....","[-1.8768295900243657, -2.079317003253634, -2.1...","[-0.1579478845925554, -0.31552721416325413, -0...","[-1.3198679406236047, -1.2157239234154333, -1....","[-2.4573798435003065, -2.5099904886228317, -2....","[-0.7119611971263433, -0.6561180597720785, -0....","[-1.4728058468702996, -1.4920091924436523, -1...."
Song2,hiphop,"[0.03420056028846946, -0.15427276804743992, -0...","[0.32253431026175666, 0.23001426150804355, 0.1...","[0.29279251046023214, 0.14362618111079553, -0....","[-0.3165794578164515, -0.6375311505150748, -0....","[-0.11063654627327543, -0.3222132224915001, -0...","[-0.5167336875877222, -0.9180172296526947, -1....","[-0.19233229411953154, -0.526762147636331, -0...."
Song3,reggae,"[0.378588476803207, 0.7300303265726806, 0.9993...","[-0.5076193006119435, -0.266691185030062, 0.10...","[-0.6964561433004302, -0.5342614824503772, -0....","[0.2184799550169764, 0.5275551495124273, 0.743...","[-0.6622229328250354, -0.5913942074626181, -0....","[-0.34765299943207595, -0.14794056778547912, 0...","[-0.3723612654285437, -0.10259514182055354, 0...."
Song4,pop,"[-0.16597226578564236, -0.4038680726331149, -0...","[-0.07288587949513724, -0.3626053279405733, -0...","[-0.887926843126093, -1.2881072580303174, -1.4...","[0.10869250865268003, -0.17703251170791157, -0...","[0.0021899155331943127, -0.4375964235466981, -...","[0.2617164148263985, -0.11697697357036468, -0....","[-0.045674600321133456, -0.4721317017692087, -..."
Song5,reggae,"[-1.2343755701533927, -0.9097082365243326, -0....","[-1.3688204203623042, -1.1505697096392704, -0....","[-0.42903583499790604, -0.20984619253812356, -...","[-1.4969524148738442, -1.048074393383775, -0.5...","[0.17186700106338373, 0.5390307470165151, 0.86...","[-1.628245438068275, -1.0897815200667755, -0.4...","[-1.4674551559051676, -0.9806868692364418, -0...."
Song6,jazz,"[-0.4020073635271416, -0.42358552337143096, -0...","[-0.8873650182748021, -0.48492317873131674, -0...","[-0.3616673335267426, -0.3799432051490359, -0....","[-0.055181682930100576, 0.015561402323139066, ...","[-1.530705499897742, -1.4504144700155959, -1.3...","[0.06767398637311037, 0.31242643281339477, 0.5...","[-0.44612771933358947, -0.22882078943810805, -..."
Song7,disco,"[0.1381713012445146, 0.21109443575182735, 0.33...","[0.1844181838534529, 0.06988659050156965, -0.0...","[1.4510048925589822, 1.344084617821756, 1.1469...","[0.7729468200221118, 1.0034680498562465, 1.203...","[0.7979103721821239, 0.7408705128985351, 0.711...","[1.6069577037923706, 1.9042274469643832, 2.060...","[1.2853145684433553, 1.424876713961773, 1.4881..."
Song8,country,"[-0.46556012642654404, -0.5300500479319714, -0...","[2.392474240951845, 2.714026759275344, 2.83109...","[0.40102883824636254, 0.40946098965636096, 0.4...","[-1.77591910251127, -1.777705347422178, -1.738...","[0.5389065354031011, 0.5624365368104443, 0.609...","[-1.9351134704994295, -1.7661080478183426, -1....","[-0.5462260547518742, -0.40317250901037716, -0..."
Song9,jazz,"[-1.7727759404546877, -1.9808893078407186, -2....","[0.3113615002029774, 0.23575827894425594, 0.13...","[-0.617383224329956, -0.5973401133044635, -0.5...","[-2.0428345802362906, -2.094622781842933, -2.0...","[0.3694707601316074, 0.1553608943738807, -0.07...","[-1.4778944746408644, -1.3354899038474322, -1....","[-1.2329043297822233, -1.2435079637928779, -1...."


In [ ]:
buf = []
for index, row in df.iterrows():
  for i in range(8):
    key = f"EEG_Ch{i+1}"
    buf.append(row[key])
  break

In [ ]:
np.shape(buf)

In [ ]:
from code.feature_extraction.feature_extraction import feature_extraction

ret = feature_extraction(buf)
ret

In [ ]:
df_cleaned

In [ ]:
df_cleaned_2

In [ ]:
# Check whether the shape seems right
print(len(df.loc["Song0", "EEG_Ch1"]))
print(len(df_cleaned.loc["Song0", "EEG_Ch1"]))
print(len(df_cleaned_2.loc["Song0", "EEG_Ch1"]))

In [ ]:
sample_rate = 250
for i in range (8): # 8 channels
  song_one = df.loc["Song0", f"EEG_Ch{i+1}"]
  song_one_cleaned = df_cleaned.loc["Song0", f"EEG_Ch{i+1}"]
  song_one_cleaned_2 = df_cleaned_2.loc["Song0", f"EEG_Ch{i+1}"]
  times = np.arange(len(song_one)) / sample_rate
  fig, (ax2, ax3) = plt.subplots(1, 2, figsize=(30, 3))
  # ax1.plot(times, song_one)
  # ax1.set_title(f"Original Signal EEG_Ch{i+1}")
  # ax1.margins(0, .1)
  # ax1.grid(alpha=.5, ls='--')

  ax2.plot(times, song_one_cleaned)
  ax2.set_title(f"Cleaned EEG_Ch{i+1}")
  ax2.margins(0, .1)
  ax2.grid(alpha=.5, ls='--')

  ax3.plot(times, song_one_cleaned_2)
  ax3.set_title(f"Cleaned EEG_Ch{i+1}")
  ax3.margins(0, .1)
  ax3.grid(alpha=.5, ls='--')
  plt.show()

In [ ]:
# For frequency filtering, all signal data has to be concatanated since we have to look at the whole session
# we therefore need to concatenate all arrays of each channel and store in a list, where each segment starts/ends

def get_concatentated_channel_data(channel_name):
  column = df[channel_name] # get the channel
  ret = []
  offsets = []
  for i in range(len(column)):
    arr = column.iloc[i]
    ret.extend(arr)
    offsets.append(len(arr))
  return ret, offsets

def get_concatentated_channel_data_cleaned(channel_name):
  column = df_cleaned[channel_name] # get the channel
  ret = []
  offsets = []
  for i in range(len(column)):
    arr = column.iloc[i]
    ret.extend(arr)
    offsets.append(len(arr))
  return ret, offsets


In [ ]:
channel_data_list = []
cleaned_channel_data_list = []
for i in range(1, 9, 1):
  concatenated_channel_data, offsets = get_concatentated_channel_data(f"EEG_Ch{i}")
  channel_data_list.append(concatenated_channel_data)
  concatenated_channel_data_cleaned, offsets_cleaned = get_concatentated_channel_data_cleaned(f"EEG_Ch{i}")
  cleaned_channel_data_list.append(concatenated_channel_data_cleaned)

In [ ]:
print(np.shape(channel_data_list))
print(np.shape(cleaned_channel_data_list))

In [ ]:
channel_data_list_np = np.array(channel_data_list)
cleaned_channel_data_list_np = np.array(cleaned_channel_data_list)

In [ ]:
# visualize channels to see whether it is denoised
sample_rate = 250
for i in range (channel_data_list_np.shape[0]):
  times = np.arange(len(channel_data_list_np[i]))/sample_rate
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(30, 3))
  ax1.plot(times, channel_data_list_np[i])
  ax1.set_title(f"Original Signal EEG_Ch{i+1}")
  ax1.margins(0, .1)
  ax1.grid(alpha=.5, ls='--')

  ax2.plot(times, cleaned_channel_data_list_np[i])
  ax2.set_title(f"Cleaned EEG_Ch{i+1}")
  ax2.margins(0, .1)
  ax2.grid(alpha=.5, ls='--')
  plt.show()

In [ ]:

for i in range(channel_data_list_np.shape[0]):
    times = np.arange(len(channel_data_list_np[i])) / sample_rate

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(30, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

    original = channel_data_list_np[i]
    cleaned = cleaned_channel_data_list_np[i]
    diff = original - cleaned

    # Plot Original and Cleaned signals
    ax1.plot(times, original, label='Original', alpha=0.6)
    ax1.plot(times, cleaned, label='Cleaned', alpha=0.6)
    ax1.set_title(f"EEG Channel {i+1}: Original vs. Cleaned")
    ax1.margins(0, .1)
    ax1.grid(alpha=.5, ls='--')
    ax1.legend()

    # Plot Difference
    ax2.plot(times, diff, label='Difference (Original - Cleaned)', linestyle='--', color='black', alpha=0.7)
    ax2.set_title("Difference")
    ax2.margins(0, .1)
    ax2.grid(alpha=.5, ls='--')
    ax2.legend()

    plt.tight_layout()
    plt.show()